# Phase 2 — Baselines (Models A, B, C)

Establishes the three reference points every later claim is measured against:

| ID | Model | Question it answers |
|---|---|---|
| **A** | `clinical_only` | How much biomarker signal is in BCVA + CST alone? |
| **B** | `oct_only` | What can the OCT image do by itself? |
| **C** | `concat_fusion` | Does ordinary feature concatenation add anything to B? |

Model D (gated fusion) is Phase 3 — it is only worth building if C shows fusion has *something*
to work with, or if C fails in a way gating would plausibly fix.

### Ground rules enforced by the code, not by discipline

- One fixed split for all three arms. Comparing across different partitions is meaningless.
- Clinical imputation, scaling and class weights fit on the **training partition only**.
- Thresholds tuned on **validation**, then frozen before test is touched.
- Test is evaluated **once per run**, at the end.

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings, time
from pathlib import Path

REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(REPO_ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from olives_biomarkers import OlivesPipeline, ExperimentSuite, ResultsAggregator
from olives_biomarkers.config import ConfigLoader
from olives_biomarkers.evaluation import ResultsPlotter

print("repo root:", REPO_ROOT)

### Choosing a training budget

`configs/local_cpu.yaml` and `configs/colab_gpu.yaml` set the same knobs at different scales.
The local profile exists so the pipeline can be exercised without a GPU — it is a **plumbing
check, not the headline experiment**, and will understate every model.

Anything that goes in the report should come from the GPU profile.

In [ ]:
pipeline = OlivesPipeline.from_config(REPO_ROOT / "configs" / "data.yaml", repo_root=REPO_ROOT)

# Pick automatically: GPU present -> full budget, otherwise the reduced one.
BUDGET = "colab_gpu" if pipeline.env.device == "cuda" else "local_cpu"
print(f"device: {pipeline.env.device}  ->  budget profile: {BUDGET}")

loader = ConfigLoader(REPO_ROOT)
budget = loader.load(REPO_ROOT / "configs" / f"{BUDGET}.yaml")
print(f"  image_size = {budget.data.image_size}")
print(f"  epochs     = {budget.training.epochs}")
print(f"  batch_size = {budget.training.batch_size}")
if BUDGET == "local_cpu":
    print("\n  NOTE: reduced budget. Treat these numbers as a pipeline check, not a result.")

In [ ]:
RUNS_DIR = REPO_ROOT / "outputs" / "runs" / BUDGET
FIGURES = REPO_ROOT / "outputs" / "figures" / "phase2"
FIGURES.mkdir(parents=True, exist_ok=True)
plotter = ResultsPlotter()

def show(figure, name):
    """Display a figure and persist it under outputs/figures/phase2."""
    if figure is None:
        print(f"(no figure for {name})")
        return
    plotter.save(figure, FIGURES / f"{name}.png")
    plt.show()

---
## 2. Data and split

The image cache turns training from a 30 GB parquet scan into 9,396 PNG reads. Export it once;
this is also the folder to upload to Drive for Colab.

In [ ]:
manifest = pipeline.get_manifest()

# One-off: export the modelling subset as PNGs (~2.3 GB). Skipped if already present.
if not pipeline.image_cache_dir.exists():
    pipeline.export_image_cache(manifest)

frame = pipeline.modelling_frame(manifest, attach_cache=True)
print(f"modelling rows: {len(frame):,}  |  patients: {frame.patient_id.nunique()}")
print(f"labels: {len(manifest.label_columns)}  ({pipeline.config.data.target_set})")
print(f"image cache: {pipeline.image_cache_dir}")

In [ ]:
assignment = pipeline.make_holdout_split(manifest, write=True)

split_table = pd.DataFrame(
    {
        "partition": list(assignment.partitions),
        "n_patients": [len(v) for v in assignment.partitions.values()],
        "n_scans": [
            int(frame.patient_id.isin(v).sum()) for v in assignment.partitions.values()
        ],
    }
)
split_table

The `calibration` partition is carved out of *train* and is patient-disjoint from everything
else, so Phase 4's temperature scaling never sees data that fitted the model weights.

---
## 3. Pre-flight checks

Two cheap tests that catch most wiring bugs before an hour of training is spent on them.

In [ ]:
from olives_biomarkers.models import ModelFactory
from olives_biomarkers.training.losses import MaskedBCEWithLogitsLoss

n_labels = len(manifest.label_columns)

# (a) Shapes, and that the head emits raw logits rather than probabilities.
for name in ["clinical_only", "oct_only", "concat_fusion"]:
    cfg = loader.load(REPO_ROOT / "configs" / "data.yaml")
    cfg.model.name = name
    cfg.model.pretrained = False
    model = ModelFactory().build(cfg.model, n_labels=n_labels, clinical_dim=4)
    model.eval()
    with torch.no_grad():
        out = model(
            image=torch.randn(4, 3, 64, 64) * 5 if model.uses_image else None,
            clinical=torch.randn(4, 4) if model.uses_clinical else None,
        )
    print(f"{name:16s} shape={tuple(out.shape)}  range=[{out.min():+.2f}, {out.max():+.2f}]  "
          f"params={model.n_parameters():,}")
assert out.min() < 0, "outputs look like probabilities - BCEWithLogitsLoss expects raw logits"

In [ ]:
# (b) Every model must be able to memorise a single batch. If it cannot, something
#     upstream of the optimiser is broken and no amount of training will help.
def overfit_one_batch(name, steps=60, lr=0.02):
    torch.manual_seed(0)
    cfg = loader.load(REPO_ROOT / "configs" / "data.yaml")
    cfg.model.name = name
    cfg.model.pretrained = False
    cfg.model.dropout = 0.0
    model = ModelFactory().build(cfg.model, n_labels=n_labels, clinical_dim=4)
    model.train()

    image = torch.randn(4, 3, 64, 64)
    clinical = torch.randn(4, 4)
    target = torch.randint(0, 2, (4, n_labels)).float()
    criterion = MaskedBCEWithLogitsLoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    for _ in range(steps):
        optimiser.zero_grad()
        loss = criterion(
            model(
                image=image if model.uses_image else None,
                clinical=clinical if model.uses_clinical else None,
            ),
            target,
        )
        loss.backward()
        optimiser.step()
        losses.append(float(loss.detach()))
    return losses

for name in ["clinical_only", "oct_only", "concat_fusion"]:
    losses = overfit_one_batch(name)
    verdict = "OK" if losses[-1] < losses[0] * 0.5 else "FAILED"
    print(f"{name:16s} loss {losses[0]:.4f} -> {losses[-1]:.4f}   {verdict}")

---
## 4. Training the three baselines

`ExperimentSuite` holds the manifest, the split and the training budget fixed and varies only
the model, so the arms stay comparable by construction.

Set `SEEDS` to several values for the real run — a single-seed difference between two models is
not evidence of anything.

In [ ]:
def make_config(model_config_name):
    """Model architecture from its own config, training budget from the profile."""
    cfg = loader.load(REPO_ROOT / "configs" / f"{model_config_name}.yaml")
    cfg.data.image_size = budget.data.image_size
    cfg.data.num_workers = budget.data.num_workers
    cfg.training.epochs = budget.training.epochs
    cfg.training.batch_size = budget.training.batch_size
    cfg.training.learning_rate = budget.training.learning_rate
    cfg.training.early_stopping_patience = budget.training.early_stopping_patience
    cfg.training.amp = budget.training.amp
    return cfg

CONFIGS = {
    "clinical_only": make_config("baseline_clinical"),
    "oct_only": make_config("baseline_oct"),
    "concat_fusion": make_config("fusion_concat"),
}
SEEDS = [42] if BUDGET == "local_cpu" else [42, 43, 44]

print(f"{len(CONFIGS)} models x {len(SEEDS)} seed(s) = {len(CONFIGS) * len(SEEDS)} runs")
for name, cfg in CONFIGS.items():
    print(f"  {name:16s} epochs={cfg.training.epochs} image={cfg.data.image_size}")

In [ ]:
RUN_TRAINING = True   # set False to skip straight to loading saved runs

if RUN_TRAINING:
    started = time.time()
    suite = ExperimentSuite(pipeline, output_root=RUNS_DIR)
    results = suite.run_models(
        configs=CONFIGS,
        assignment=assignment,
        manifest=manifest,
        seeds=SEEDS,
    )
    print(f"\ntrained {len(results)} runs in {(time.time() - started) / 60:.1f} min")
else:
    from olives_biomarkers import RunResult
    results = RunResult.load_all(RUNS_DIR)
    print(f"loaded {len(results)} saved runs from {RUNS_DIR}")

---
## 5. Results

In [ ]:
aggregator = ResultsAggregator(results)
comparison = aggregator.comparison(sort_by="macro_auprc")
comparison[
    ["model", "seed", "n_parameters", "epochs_run", "minutes",
     "macro_f1", "macro_auroc", "macro_auprc", "n_degenerate_labels"]
]

In [ ]:
show(plotter.model_comparison(comparison), "01_model_comparison")

In [ ]:
histories = {r.model_name: r.history.to_frame() for r in results if r.seed == SEEDS[0]}
show(plotter.training_curves(histories, monitor="val_macro_auprc"), "02_training_curves")

Watch for two failure modes in these curves:

- **Validation loss rising while training loss falls** — overfitting, expected with 52 training
  patients. Early stopping should catch it; check `epochs_run` above.
- **A flat monitored metric** — the model learned nothing, usually a learning-rate or
  class-weight problem rather than a capacity one.

In [ ]:
if len(SEEDS) > 1:
    display(aggregator.across_seeds())
else:
    print("Single seed - no spread to report. Run several seeds before comparing models.")

---
## 6. Per-label results

The macro average hides the thing that matters most here: performance is bimodal, split between
the common biomarkers and the rare ones that barely appear in the test partition.

In [ ]:
per_label = aggregator.per_label_pivot(metric="auprc")
per_label

In [ ]:
show(plotter.per_label_comparison(per_label, metric="auprc"), "03_per_label_auprc")

In [ ]:
# Which labels could not be scored at all, and why.
undefined = aggregator.per_label()
undefined = undefined[undefined["degenerate"]][["run_id", "model", "label", "n_positive"]]
if len(undefined):
    display(undefined.drop_duplicates(["model", "label"]))
    print("\nThese labels have no positive examples in the test partition. Their metrics are "
          "undefined and are reported as NaN, never as 0.")
else:
    print("Every label had positives in the test partition.")

---
## 7. Reading the baselines

The three questions this phase exists to answer:

In [ ]:
means = comparison.groupby("model")[["macro_f1", "macro_auroc", "macro_auprc"]].mean()
display(means.round(4))

clinical = means.loc["clinical_only"]
oct_only = means.loc["oct_only"]
concat = means.loc["concat_fusion"]

print("\n1. How much lives in BCVA + CST alone?")
print(f"   clinical-only macro AUROC = {clinical['macro_auroc']:.4f} "
      f"({'above' if clinical['macro_auroc'] > 0.55 else 'near'} chance)")

print("\n2. Does the image beat the clinical baseline?")
delta = oct_only["macro_auprc"] - clinical["macro_auprc"]
print(f"   OCT - clinical macro AUPRC = {delta:+.4f}")

print("\n3. Does plain concatenation add anything to the image?")
delta = concat["macro_auprc"] - oct_only["macro_auprc"]
print(f"   concat - OCT macro AUPRC   = {delta:+.4f}")
print("\n   A small or negative difference here does not close the question: it is exactly the")
print("   case gated fusion is meant to address, since concatenation cannot express an")
print("   interaction between the modalities.")

In [ ]:
# Where does the clinical baseline do best? Compare against the EDA prediction that
# CST tracks the fluid biomarkers.
clinical_runs = [r for r in results if r.model_name == "clinical_only"]
if clinical_runs and clinical_runs[0].per_label is not None:
    table = clinical_runs[0].per_label[["label", "n_positive", "auroc", "auprc"]]
    display(table.sort_values("auroc", ascending=False).head(8))
    print("\nEDA predicted CST would separate the fluid biomarkers (irf, drt_me, srf) and be")
    print("uninformative about the vitreous-face ones (pavf, favf). Check that ordering above.")

---
## 8. Findings and next step

**What is settled**

- The three baselines are trained on one fixed patient-grouped split, with thresholds fitted on
  validation and frozen before test.
- Per-label results show which biomarkers are measurable at all on this cohort.

**What is not settled**

- Whether any difference between models is real. That needs several seeds and the overlapping
  confidence intervals from Phase 5 — a gap in point estimates is not a result.
- Whether clinical features help *conditionally* on the image. Concatenation only tests one very
  restricted form of fusion.

**Next:** `03_gated_fusion.ipynb` — Model D, plus the ablations that tell us whether the gate is
doing anything at all.